In [317]:
qtd_fornecedores = 200
meses = 24
tx_anomalia_mes = 0.06
seed = 42

In [318]:
import pandas as pd
import numpy as np
import random

random.seed(seed)
np.random.seed(seed)

In [319]:
gp_crescente = ['crescente'] * 80
gp_decrescente = ['decrescente'] * 60
gp_estavel = ['estavel'] * 60

grupos = gp_crescente + gp_decrescente + gp_estavel
random.shuffle(grupos)

In [320]:
fornecedores = []

for i in range(200):
    opcao = grupos[i]
    padroes = ['NUM', 'ACC', 'FORN']

    padrao_atual = random.choice(padroes)

    if padrao_atual == 'NUM':
        id_fornecedor = f'{1000000+i}'
    
    elif padrao_atual == 'ACC':
        id_fornecedor = f'ACC-{i+1:05d}'

    elif padrao_atual == 'FORN':
        id_fornecedor = f'FORN-{i+1:05d}'
    
    valor_medio_inicial = round(np.random.lognormal(8, 0.6), 2)

    if valor_medio_inicial <= 4000:
        freq_media_inicial = np.random.randint(10, 19)
    elif valor_medio_inicial > 4000 and valor_medio_inicial <= 10000:
        freq_media_inicial = np.random.randint(6, 13)
    elif valor_medio_inicial > 10000:
        freq_media_inicial = np.random.randint(3, 9)

    prob_inicio = np.random.uniform(0, 1)

    if prob_inicio < 0.6:
        mes_inicio = np.random.randint(1, 4)
    else:
        mes_inicio = np.random.randint(4, 24)

    if mes_inicio > 18:
        mes_fim = 24
    else:
        prob_fim = np.random.uniform(0, 1)
        if prob_fim < 0.6:
            mes_fim = 24
        else:
            mes_fim = np.random.randint(mes_inicio + 6, 25)

    fornecedores.append({
        'id_fornecedor_raw': id_fornecedor,
        'grupo': opcao,
        'valor_medio_inicial': valor_medio_inicial,
        'freq_media_inicial': freq_media_inicial,
        'mes_inicio': mes_inicio,
        'mes_fim': mes_fim
    })
df_fornecedores = pd.DataFrame(fornecedores)

print(mes_inicio, mes_fim)

#print(df_fornecedores)

3 19


In [323]:
df_calendario = pd.DataFrame({'data': pd.RangeIndex(start=1, stop=25, step=1)})

merge_df = df_fornecedores.merge(df_calendario, how='cross')

model_temporal = merge_df.loc[(merge_df['data']>=merge_df['mes_inicio']) & (merge_df['data']<=merge_df['mes_fim'])]

print(model_temporal)

     id_fornecedor_raw      grupo  valor_medio_inicial  freq_media_inicial  \
9              1000000  crescente              4015.95                   8   
10             1000000  crescente              4015.95                   8   
11             1000000  crescente              4015.95                   8   
12             1000000  crescente              4015.95                   8   
13             1000000  crescente              4015.95                   8   
...                ...        ...                  ...                 ...   
4790           1000199    estavel              2483.11                  13   
4791           1000199    estavel              2483.11                  13   
4792           1000199    estavel              2483.11                  13   
4793           1000199    estavel              2483.11                  13   
4794           1000199    estavel              2483.11                  13   

      mes_inicio  mes_fim  data  
9             10       24    

In [327]:
taxa = 0.015
model_temporal['mes_relativo'] = model_temporal['data'] - model_temporal['mes_inicio']
model_temporal['lambda_mes'] = model_temporal['freq_media_inicial']
print(model_temporal)

if 'crescente' in model_temporal['grupo'].values:
    lambda_mes = freq_media_inicial * (1 + taxa) ** model_temporal['mes_relativo']
elif 'decrescente' in model_temporal['grupo'].values:
    (1 - taxa) ** model_temporal['mes_relativo']

print(model_temporal)

     id_fornecedor_raw      grupo  valor_medio_inicial  freq_media_inicial  \
9              1000000  crescente              4015.95                   8   
10             1000000  crescente              4015.95                   8   
11             1000000  crescente              4015.95                   8   
12             1000000  crescente              4015.95                   8   
13             1000000  crescente              4015.95                   8   
...                ...        ...                  ...                 ...   
4790           1000199    estavel              2483.11                  13   
4791           1000199    estavel              2483.11                  13   
4792           1000199    estavel              2483.11                  13   
4793           1000199    estavel              2483.11                  13   
4794           1000199    estavel              2483.11                  13   

      mes_inicio  mes_fim  data  mes_relativo  lambda_mes  
9  